# Graph-Augmented Intelligence — Webinar
## Notebook 02: GDS Algorithms in Neo4j Aura

**This notebook is a reference guide.** Run these commands in the
**Neo4j Aura Workspace** (Query tab) — not in Databricks.

After running `01_neo4j_ingest`, switch to Aura and execute the steps below.
When finished, return to Databricks and run `03_pull_and_model`.

---

### What We're Computing

| Algorithm | Property Written | Fraud Signal |
|-----------|-----------------|-------------|
| **PageRank** | `Account.risk_score` | Central accounts in money-flow networks |
| **Louvain** | `Account.community_id` | Tightly connected clusters (fraud rings) |
| **Node Similarity** | `Account.similarity_score` | Accounts sharing the same merchants |

---
## Step 1: Verify the Graph Loaded Correctly

Run in Aura Workspace → Query tab:

```cypher
// Check node and relationship counts
MATCH (a:Account) WITH count(a) AS accounts
MATCH (m:Merchant) WITH accounts, count(m) AS merchants
MATCH ()-[t:TRANSACTED_WITH]->() WITH accounts, merchants, count(t) AS txns
MATCH ()-[p:TRANSFERRED_TO]->() WITH accounts, merchants, txns, count(p) AS p2p
RETURN accounts, merchants, txns, p2p
```

**Expected:** ~5,000 accounts, ~500 merchants, ~50,000 transactions, ~8,000 transfers.

---
## Step 2: Project the Account Transfer Graph

GDS algorithms run on an **in-memory graph projection**, not directly on the database.
This projects only Account nodes and TRANSFERRED_TO relationships — the peer-to-peer
money-flow graph where fraud rings live.

```cypher
CALL gds.graph.project(
  'account_transfers',
  'Account',
  {TRANSFERRED_TO: {orientation: 'UNDIRECTED'}}
)
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
```

**Expected:** ~5,000 nodes, ~8,000 relationships.

---
## Step 3: Run PageRank (Risk Centrality)

PageRank measures how "central" an account is in the transfer network.
Accounts that receive money from many well-connected accounts score higher —
exactly how money-mule networks operate.

```cypher
CALL gds.pageRank.write(
  'account_transfers',
  {
    maxIterations: 20,
    dampingFactor: 0.85,
    writeProperty: 'risk_score'
  }
)
YIELD nodePropertiesWritten, ranIterations, didConverge
RETURN nodePropertiesWritten, ranIterations, didConverge
```

**Verify — top 10 by PageRank:**
```cypher
MATCH (a:Account)
WHERE a.risk_score IS NOT NULL
RETURN a.account_id AS id,
       a.is_fraud AS fraud,
       round(a.risk_score, 6) AS pagerank
ORDER BY a.risk_score DESC
LIMIT 10
```

Look for fraud accounts appearing in the top results — that's the signal.

---
## Step 4: Run Louvain Community Detection (Fraud Rings)

Louvain finds clusters of densely connected accounts. In a legitimate network,
communities are large and diffuse. Fraud rings form **small, tight clusters**
with heavy internal transfers.

```cypher
CALL gds.louvain.write(
  'account_transfers',
  {
    writeProperty: 'community_id'
  }
)
YIELD communityCount, modularity, nodePropertiesWritten
RETURN communityCount, modularity, nodePropertiesWritten
```

**Verify — community size distribution:**
```cypher
MATCH (a:Account)
WHERE a.community_id IS NOT NULL
RETURN a.community_id AS community, count(*) AS size
ORDER BY size DESC
LIMIT 15
```

**Visualise a fraud-heavy community:**
```cypher
MATCH (a:Account)
WHERE a.community_id IS NOT NULL AND a.is_fraud = true
WITH a.community_id AS community, count(*) AS fraud_count
ORDER BY fraud_count DESC LIMIT 1
WITH community
MATCH (m:Account {community_id: community})-[r:TRANSFERRED_TO]-(other:Account {community_id: community})
RETURN m, r, other
```

---
## Step 5: Drop the Transfer Graph Projection

Clean up before creating the next projection.

```cypher
CALL gds.graph.drop('account_transfers')
YIELD graphName
RETURN graphName
```

---
## Step 6: Project the Bipartite Graph (Account → Merchant)

Node Similarity needs the bipartite graph: which accounts transact
with which merchants.

```cypher
CALL gds.graph.project(
  'account_merchants',
  ['Account', 'Merchant'],
  {TRANSACTED_WITH: {orientation: 'NATURAL'}}
)
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
```

---
## Step 7: Run Node Similarity (Shared Merchant Patterns)

Two accounts are similar if they transact with the **same merchants**.
Fraud accounts typically share a small set of high-risk merchants.

```cypher
CALL gds.nodeSimilarity.write(
  'account_merchants',
  {
    similarityMetric: 'JACCARD',
    topK: 5,
    similarityCutoff: 0.3,
    writeRelationshipType: 'SIMILAR_TO',
    writeProperty: 'similarity_score'
  }
)
YIELD nodesCompared, relationshipsWritten
RETURN nodesCompared, relationshipsWritten
```

**Verify — most similar account pairs:**
```cypher
MATCH (a:Account)-[s:SIMILAR_TO]-(b:Account)
WHERE a.account_id < b.account_id
RETURN a.account_id AS account_a,
       b.account_id AS account_b,
       round(s.similarity_score, 3) AS similarity,
       a.is_fraud AS a_fraud,
       b.is_fraud AS b_fraud
ORDER BY s.similarity_score DESC
LIMIT 10
```

---
## Step 8: Aggregate Max Similarity per Account

For each account, store its **highest similarity score** as a node property.
This makes it easy to read back as a single feature column in Databricks.

```cypher
MATCH (a:Account)
OPTIONAL MATCH (a)-[s:SIMILAR_TO]-()
WITH a, COALESCE(MAX(s.similarity_score), 0.0) AS max_sim
SET a.similarity_score = max_sim
RETURN count(a) AS accounts_updated
```

---
## Step 9: Drop the Bipartite Graph Projection

```cypher
CALL gds.graph.drop('account_merchants')
YIELD graphName
RETURN graphName
```

---
## Step 10: Final Verification — All Features Written

Confirm all three properties exist on Account nodes:

```cypher
MATCH (a:Account)
WHERE a.risk_score IS NOT NULL
  AND a.community_id IS NOT NULL
  AND a.similarity_score IS NOT NULL
RETURN count(a) AS accounts_with_all_features
```

**Fraud vs legitimate comparison:**
```cypher
MATCH (a:Account)
RETURN a.is_fraud AS is_fraud,
       count(a) AS count,
       round(avg(a.risk_score), 6) AS avg_pagerank,
       round(avg(a.similarity_score), 4) AS avg_similarity,
       count(DISTINCT a.community_id) AS num_communities
ORDER BY a.is_fraud
```

You should see clear separation between fraud and legitimate accounts
across all three features.

---
## Done in Aura!

The graph now has three GDS-computed properties on every Account node:
- `risk_score` — centrality in the transfer network
- `community_id` — Louvain cluster assignment
- `similarity_score` — highest Jaccard similarity to any other account

**Next →** Return to Databricks and run `03_pull_and_model` to read these
features back and measure the ML lift.